In [ ]:
# bootstrap: Colab (private clone via GH_TOKEN) + local import of `agentcore` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret; add a GitHub token (repo scope) as Colab secret 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_repo)], check=True)
        subprocess.run(["git", "-C", str(_repo), "remote", "set-url", "origin", f"https://github.com/{_slug}.git"])
    os.chdir(_repo / "07-application-agent-framework/agent-fundamentals/agent-core")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "agentcore").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 02 · Tools are contracts

A tool is a function the model may call — but the model is a reader that cannot ask
follow-up questions and will happily guess. So a good tool removes the need to guess:
a clear schema, arguments validated **before** running, and errors the model can act
on instead of stack traces.

In this notebook you write tools the right way and see how the loop reacts to each
kind of result.

> **Exercise cells** contain `# YOUR CODE HERE` — replace it, then run the **Check** cell below it. A check prints ✅ when it passes. The finished version is in `solutions/`.

In [ ]:
from agentcore import Agent, FakeLLM, ToolError, call, tool

@tool
def get_order(order_id: str) -> dict:
    """Look up an order by id."""
    orders = {"ORD-1": {"status": "shipped", "total": 42.0}}
    if order_id not in orders:
        raise ToolError(f"no order {order_id}", kind="not_found",
                        hint="Ask the customer to confirm the order id.")
    return {"order_id": order_id, **orders[order_id]}

print("schema     :", get_order.schema)
print("good call  :", get_order.run({"order_id": "ORD-1"}))
print("not found  :", get_order.run({"order_id": "ORD-9"}))
print("bad args   :", get_order.run({}))                      # missing required argument

Notice the three result shapes the model can receive:
`{"ok": True, "data": ...}`, a **structured error** with a `hint`, and an
`invalid_arguments` error. None of them is a crash. That is what lets the model
recover — ask for the missing id, apologise for the not-found — instead of the whole
turn failing.

## Exercise 2.1 — a tool with an enum-like check

Write `set_priority(ticket_id: str, level: str)` that accepts only `"low"`, `"medium"`
or `"high"`. On a bad level, raise `ToolError(..., kind="invalid_value", hint=...)`.
On success return `{"ticket_id": ..., "level": ...}`.

In [ ]:
@tool
def set_priority(ticket_id: str, level: str) -> dict:
    """Set a ticket's priority to low, medium, or high."""
    # YOUR CODE HERE
    raise NotImplementedError("your turn")

In [ ]:
assert set_priority.run({"ticket_id": "T1", "level": "high"})["data"]["level"] == "high"
bad = set_priority.run({"ticket_id": "T1", "level": "urgent"})
assert bad["ok"] is False and bad["error"] == "invalid_value" and bad["hint"]
assert set_priority.run({"ticket_id": "T1"})["error"] == "invalid_arguments"
print("✅ set_priority validates its input")

## Exercise 2.2 — make a write idempotent

Writes get retried (a timeout, a dropped connection). A retried "refund" must not
refund twice. Implement `make_refund_tool()` returning a `@tool`-decorated function
`refund(order_id, amount)` that records each `order_id` it has refunded in a closure
set; a second call for the same order returns the **same** result with
`{"already_done": True}` and does **not** append again.

In [ ]:
def make_refund_tool():
    done = {}                         # order_id -> result, the idempotency record
    @tool
    def refund(order_id: str, amount: float) -> dict:
        """Refund an order. Safe to retry."""
        # YOUR CODE HERE
        raise NotImplementedError("your turn")
    return refund

In [ ]:
refund = make_refund_tool()
first = refund.run({"order_id": "ORD-1", "amount": 20.0})
again = refund.run({"order_id": "ORD-1", "amount": 20.0})
assert first["data"]["refunded"] == 20.0
assert again["data"].get("already_done") is True
print("✅ refund is idempotent:", first["data"], "→", again["data"])

## Exercise 2.3 — see the loop recover from a tool error

Give an `Agent` the `get_order` tool and a **scripted model** that: (1) calls
`get_order` for a missing order `ORD-9`, then (2) after seeing the `not_found` error,
answers with the text `"I couldn't find order ORD-9 — can you confirm the number?"`.
Build the `FakeLLM` script in `script` so the assertion passes.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
r = Agent(FakeLLM(script), tools=[get_order]).run("where is my order ORD-9?")
assert r.done and "confirm" in r.text.lower()
tool_result = [m for m in r.messages if m["role"] == "tool"][0]["content"]
assert "not_found" in tool_result
print(r.transcript())
print("✅ the agent recovered from a not-found error instead of crashing")

## The one-minute version
In a design round, when you "design the interface", write one full tool contract on
the board: name, description, arguments with an enum, the success shape, the error
cases, and an idempotency key for writes. One concrete contract beats a list of tool
names — it shows you have thought about what the model will do wrong.